# Lecția 01 - Introducere în Agenții AI

Bine ați venit la prima lecție din cursul **Agenți AI pentru Începători**!

Un **agent AI** este un program care folosește un model lingvistic mare (LLM) ca motor de raționament și poate lua *acțiuni* în lumea reală — apelând API-uri, interogând baze de date sau rulând cod — pentru a îndeplini un scop în numele unui utilizator.

În acest notebook veți construi primul vostru agent: un **Agent de Călătorie** care recomandă destinații de vacanță. Pe parcurs veți învăța cum să:

1. Vă conectați la Azure AI Foundry Agent Service folosind **Microsoft Agent Framework**.
2. Oferiți agentului un **instrument** — o funcție simplă Python pe care o poate apela.
3. Rulați agentul și să inspectați răspunsul său.
4. Transmiteți răspunsul agentului token cu token.


## Configurare

Înainte de a rula acest notebook, asigură-te că ai:

1. **Un proiect Azure AI Foundry** cu un model de chat implementat (de ex. `gpt-4o-mini`).
2. **Autentificat cu Azure CLI** — rulează `az login` în terminalul tău.
3. **Setat variabilele de mediu necesare:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint-ul proiectului tău Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — numele modelului tău implementat.

Celula de mai jos instalează pachetele Python de care ai nevoie.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Crearea primului tău agent

Un agent are nevoie de două lucruri:

- **Instrucțiuni** care îi spun *cine este* și *cum să se comporte* (un prompt de sistem).
- **Instrumente** — funcții Python decorate cu `@tool` pe care agentul le poate apela pentru a obține informații sau a efectua acțiuni.

Mai jos definim un instrument simplu care returnează o listă de destinații populare de vacanță. Agentul va folosi acest instrument atunci când un utilizator cere recomandări de călătorie.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Pentru o experiență mai interactivă, poți **transmite în flux** răspunsul agentului. În loc să aștepți răspunsul complet, agentul livrează bucăți de text pe măsură ce sunt generate. Acest lucru este deosebit de util în interfețele de chat unde dorești să afișezi ieșirea în timp real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Rezumat

În această lecție ai învățat cum să:

- **Creezi un furnizor** care se conectează la Azure AI Foundry Agent Service prin `AzureAIProjectAgentProvider`.
- **Defini o unealtă** folosind decoratul `@tool` astfel încât agentul să poată apela funcțiile tale Python.
- **Rulezi agentul** cu un mesaj de la utilizator și să afișezi răspunsul său.
- **Redai fluxuri de răspunsuri** pentru output în timp real.

În următoarea lecție vom explora cadre agentice mai în detaliu și vom învăța cum să oferim agenților unelte mai puternice și capacități de raționament în mai mulți pași.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertisment**:
Acest document a fost tradus folosind serviciul de traducere automată AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original, în limba sa nativă, trebuie considerat sursa autoritară. Pentru informații critice, este recomandată traducerea profesională realizată de un specialist uman. Nu ne asumăm nicio responsabilitate pentru neînțelegeri sau interpretări greșite rezultate din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
